<center><h1>Hetrograph Visualization</h1></center></br>

</br><p>Visualizing heterogeneous graph data enables a deeper understanding of complex networks composed of multiple node and edge types. In a 2D representation, users can clearly identify patterns such as clusters, central nodes, and relationship paths, making it highly effective for quick analysis and presentation.</p>

<p>On the other hand, 3D visualization offers a more immersive and interactive experience, particularly beneficial for exploring dense or layered graphs. It allows users to rotate, zoom, and navigate through the graph space, helping uncover hidden structures or overlapping connections that are difficult to perceive in 2D.</p></br>


In [1]:
import plotly.graph_objects as go
import networkx as nx
import torch

In [2]:
node_type_colors = {}
edge_type_colors = {}


def plot_style(hetrograph):

    global node_type_colors, edge_type_colors
    
    heteroData = hetrograph.subgraph({'case': torch.arange(50)})

    G = nx.DiGraph()

    node_type_colors = {
        'case': 'skyblue',
        'subtype': 'pink',
        'gene': 'green',
        'protein': 'orange',
        'cnv': 'purple',
        'mutation': 'red',
    }

    node_pos_map = {}
    for node_type in heteroData.node_types:
        for i in range(heteroData[node_type].num_nodes):
            node_id = f'{node_type}_{i}'
            G.add_node(node_id, node_type=node_type)
            node_pos_map[node_id] = node_type_colors[node_type]

    edge_type_colors = {
        ('gene', 'in', 'case'): 'gray',
        ('protein', 'in', 'case'): 'black',
        ('cnv', 'in', 'case'): 'brown',
        ('mutation', 'in', 'case'): 'crimson',
        ('case', 'similar_to', 'case'): 'blue',
        ('case', 'coexpr_with', 'case'): 'cyan',
        ('gene', 'interacts', 'gene'): 'limegreen',
        ('protein', 'ppi', 'protein'): 'darkorange',
        ('cnv', 'co_cnv', 'cnv'): 'violet',
        ('mutation', 'co_mutation', 'mutation'): 'pink',
        ('gene', 'encodes', 'protein'): 'gold',
        ('mutation', 'in_gene', 'gene'): 'lightcoral',
        ('cnv', 'affects', 'gene'): 'darkmagenta',
        ('case', 'has_subtype', 'subtype'): 'navy',
        ('subtype', 'characterized_by', 'gene'): 'olive',
    }

    for edge_type in heteroData.edge_types:
        src_type, rel_type, dst_type = edge_type
        edges = heteroData[edge_type].edge_index
        src_nodes = edges[0].tolist()
        dst_nodes = edges[1].tolist()

        for src, dst in zip(src_nodes, dst_nodes):
            src_id = f'{src_type}_{src}'
            dst_id = f'{dst_type}_{dst}'
            G.add_edge(src_id, dst_id, edge_type=rel_type,
                       color=edge_type_colors.get(edge_type, 'gray'))

    return G

</br><h4>2D Visualization:</h4>

In [3]:
def hetrograph_2d_visualization(hetrograph):
    
    global node_type_colors, edge_type_colors

    G = plot_style(hetrograph)
            
    pos = nx.spring_layout(G, seed=42)
    
    node_traces = {}
    for node, data in G.nodes(data=True):
        node_type = data['node_type']
        color = node_type_colors[node_type]
        if node_type not in node_traces:
            node_traces[node_type] = {'x': [], 'y': [], 'text': [], 'color': color}
        x, y = pos[node]
        node_traces[node_type]['x'].append(x)
        node_traces[node_type]['y'].append(y)
        node_traces[node_type]['text'].append(node)
    
    edge_traces = {}
    for u, v, data in G.edges(data=True):
        src_type = G.nodes[u]['node_type']
        dst_type = G.nodes[v]['node_type']
        edge_label = data['edge_type']
        color = edge_type_colors.get((src_type, edge_label, dst_type), 'gray')
        key = f"{src_type} → {dst_type} ({edge_label})"
        if key not in edge_traces:
            edge_traces[key] = {'x': [], 'y': [], 'color': color}
        edge_traces[key]['x'] += [pos[u][0], pos[v][0], None]
        edge_traces[key]['y'] += [pos[u][1], pos[v][1], None]
    
    fig = go.Figure()
    
    # Add node traces
    for node_type, trace in node_traces.items():
        fig.add_trace(go.Scatter(
            x=trace['x'], y=trace['y'],
            mode='markers',
            marker=dict(color=trace['color'], size=6),
            text=trace['text'],
            name=f"Node: {node_type}",
            legendgroup=f"node_{node_type}",
            showlegend=True
        ))
    
    # Add edge traces
    for edge_label, trace in edge_traces.items():
        fig.add_trace(go.Scatter(
            x=trace['x'], y=trace['y'],
            mode='lines',
            line=dict(color=trace['color'], width=1),
            name=f"Edge: {edge_label}",
            legendgroup=f"edge_{edge_label}",
            showlegend=True
        ))
    
    fig.update_layout(
        title="Interactive Heterogeneous Graph",
        title_x=0.5,
        hovermode='closest',
        legend_title="Toggle Node/Edge Types\n",
        width=1000,
        height=800,
        margin=dict(l=10, r=10, t=40, b=10),
        showlegend=True
    )
    
    fig.show()


</br><h4>3D Visualization:</h4>

In [4]:
def hetrograph_3d_visualization(hetrograph):

    global node_type_colors, edge_type_colors

    G = plot_style(hetrograph)
    
    pos = nx.spring_layout(G, dim=3, seed=42)

    node_traces = {}
    for node, data in G.nodes(data=True):
        node_type = data['node_type']
        color = node_type_colors[node_type]
        if node_type not in node_traces:
            node_traces[node_type] = {'x': [], 'y': [], 'z': [], 'text': [], 'color': color}
        x, y, z = pos[node]
        node_traces[node_type]['x'].append(x)
        node_traces[node_type]['y'].append(y)
        node_traces[node_type]['z'].append(z)
        node_traces[node_type]['text'].append(node)

    edge_traces = {}
    for u, v, data in G.edges(data=True):
        src_type = G.nodes[u]['node_type']
        dst_type = G.nodes[v]['node_type']
        edge_label = data['edge_type']
        color = edge_type_colors.get((src_type, edge_label, dst_type), 'gray')
        key = f"{src_type} → {dst_type} ({edge_label})"
        if key not in edge_traces:
            edge_traces[key] = {'x': [], 'y': [], 'z': [], 'color': color}
        x0, y0, z0 = pos[u]
        x1, y1, z1 = pos[v]
        edge_traces[key]['x'] += [x0, x1, None]
        edge_traces[key]['y'] += [y0, y1, None]
        edge_traces[key]['z'] += [z0, z1, None]

    fig = go.Figure()

    # Add node traces
    for node_type, trace in node_traces.items():
        fig.add_trace(go.Scatter3d(
            x=trace['x'], y=trace['y'], z=trace['z'],
            mode='markers',
            marker=dict(size=4, color=trace['color']),
            text=trace['text'],
            name=f"Node: {node_type}",
            legendgroup=f"node_{node_type}",
            showlegend=True
        ))

    # Add edge traces
    for edge_label, trace in edge_traces.items():
        fig.add_trace(go.Scatter3d(
            x=trace['x'], y=trace['y'], z=trace['z'],
            mode='lines',
            line=dict(color=trace['color'], width=1),
            name=f"Edge: {edge_label}",
            legendgroup=f"edge_{edge_label}",
            showlegend=True
        ))

    fig.update_layout(
        title="3D Interactive Heterogeneous Graph",
        title_x=0.5,
        scene=dict(
            xaxis=dict(showgrid=False, zeroline=False),
            yaxis=dict(showgrid=False, zeroline=False),
            zaxis=dict(showgrid=False, zeroline=False),
        ),
        legend_title="Toggle Node/Edge Types",
        width=1000,
        height=800,
        margin=dict(l=10, r=10, t=40, b=10),
        showlegend=True
    )

    fig.show()
